In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
from sklearn.cluster import DBSCAN
from prettytable import PrettyTable

# =========================
# Konfiguration
# =========================
original_image_path = ......................................................................
# Transparentes Overlay (wird erzeugt)
out_png = ..................................................................................
# Overlay als Umriss auf Mask2
out_jpg = ..................................................................................

# NEU: Hintergrundbild (Original2) – ohne oder mit Endung möglich
background_image_hint = ....................................................................
# NEU: zusammengesetztes Ergebnis
out_all = ..................................................................................

# Vorverarbeitung / Schwellen
GAUSS_BLUR_KSIZE = (9, 9)
ADAPTIVE_BLOCKSIZE = 11     # ungerade > 1
ADAPTIVE_C = 2
THRESH_INVERT = False       # True setzen, falls Spots dunkler als Hintergrund

# Kontur-/Blob-Filter (Größe & Form)
AUTO_AREA_FILTER = True
AREA_P_LOW = 65.0           # behält größere Spots
AREA_P_HIGH = 99.5          # kappt extreme Ausreißer
MANUAL_AREA_MIN = 20.0      # nur relevant, wenn AUTO_AREA_FILTER=False
MANUAL_AREA_MAX = None
CIRCULARITY_MIN = 0.60      # 4πA/P^2 ∈ [0..1]; höher = runder

# DBSCAN-Parameter (Pixel!)
DBSCAN_EPS = 123
DBSCAN_MIN_SAMPLES = 3

# Distanzmetrik für NN: "euclid" (empfohlen) oder "manhattan"
DIST_METRIC = "euclid"

# Radius-Policy der Schutzkreise:
# - "second": r = SCALE * d2 + OFFSET
# - "mean2" : r = SCALE * 0.5*(d1+d2) + OFFSET
RADIUS_POLICY = "second"
SCALE = 1.00
OFFSET = 0.0
MIN_RADIUS = 0.0
MAX_RADIUS = None

# Cluster-Auswahl: "center" (nächster zur Bildmitte) oder "largest" (meiste Punkte)
CLUSTER_SELECT = "center"

# Zeichnung
EDGE_COLOR_BGR = (0, 0, 255)
EDGE_THICKNESS = 2
FILL_COLOR_BGRA = (169, 188, 238, 255)
DRAW_FILLED_ON_PNG = True

# =========================
# Hilfsfunktionen
# =========================

def log(msg: str):
    print(msg, flush=True)

def safe_imwrite(path: str, img: np.ndarray) -> bool:
    """Robustes Speichern: versucht zuerst cv2.imwrite, fällt bei Bedarf auf Pillow zurück."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    try:
        ok = cv2.imwrite(path, img)
        if ok:
            return True
    except Exception as e:
        print(f"cv2.imwrite Exception for {path}: {e}")
    try:
        from PIL import Image
        if img.ndim == 2:
            Image.fromarray(img).save(path)
        elif img.shape[2] == 3:
            Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).save(path)
        elif img.shape[2] == 4:
            Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA)).save(path)
        else:
            raise ValueError(f"Unsupported image shape for Pillow: {img.shape}")
        return True
    except Exception as e:
        print(f"Pillow save failed for {path}: {e}")
        return False

def resolve_image_path(hint: str):
    """Nimmt einen Pfad mit/ohne Endung und gibt einen existierenden Pfad zurück oder None."""
    p = Path(hint)
    if p.suffix:
        return str(p) if p.exists() else None
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]
    for ext in exts:
        cand = str(p.with_suffix(ext))
        if Path(cand).exists():
            return cand
    return None

def preprocess(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if GAUSS_BLUR_KSIZE and GAUSS_BLUR_KSIZE[0] > 0:
        gray = cv2.GaussianBlur(gray, GAUSS_BLUR_KSIZE, 0)
    thresh_flag = cv2.THRESH_BINARY_INV if THRESH_INVERT else cv2.THRESH_BINARY
    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresh_flag,
        ADAPTIVE_BLOCKSIZE, ADAPTIVE_C
    )
    return gray, thresh

def contours_centroids_area_circularity(thresh):
    contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    centroids, areas, circs, kept = [], [], [], []
    for c in contours:
        A = cv2.contourArea(c)
        if A <= 0: continue
        P = cv2.arcLength(c, True)
        circ = 4.0 * np.pi * A / (P * P + 1e-9)  # Circularity
        M = cv2.moments(c)
        if M["m00"] <= 0: continue
        cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
        centroids.append((cx, cy)); areas.append(A); circs.append(circ); kept.append(c)
    return (np.array(centroids, float),
            np.array(areas, float),
            np.array(circs, float),
            kept)

def apply_size_shape_filter(centroids, areas, circs):
    if len(centroids) == 0:
        return centroids, np.ones(0, dtype=bool)
    if AUTO_AREA_FILTER:
        lo = np.percentile(areas, AREA_P_LOW)
        hi = np.percentile(areas, AREA_P_HIGH)
        area_mask = (areas >= lo) & (areas <= hi)
        log(f"    Auto-Filter: AREA_P_LOW={AREA_P_LOW:.1f}%→{lo:.1f}px², AREA_P_HIGH={AREA_P_HIGH:.1f}%→{hi:.1f}px²")
    else:
        lo = MANUAL_AREA_MIN if MANUAL_AREA_MIN is not None else -np.inf
        hi = MANUAL_AREA_MAX if MANUAL_AREA_MAX is not None else np.inf
        area_mask = (areas >= lo) & (areas <= hi)
        log(f"    Manual-Filter: area ∈ [{lo}, {hi}] px²")
    circ_mask = (circs >= CIRCULARITY_MIN)
    mask = area_mask & circ_mask
    log(f"    Größe ok: {area_mask.sum()} | Circularity ok: {circ_mask.sum()} | gemeinsam: {mask.sum()}")
    return centroids[mask], mask

def pairwise_distances(X, metric="euclid"):
    if len(X) == 0: return np.empty((0, 0), float)
    diffs = X[:, None, :] - X[None, :, :]
    if metric == "manhattan":
        D = np.abs(diffs).sum(axis=2)
    else:
        D = np.linalg.norm(diffs, axis=2)
    np.fill_diagonal(D, np.inf)
    return D

def two_nn_distances(X, metric="euclid"):
    N = len(X)
    if N == 0: return np.array([]), np.array([])
    if N == 1: return np.array([0.0]), np.array([0.0])
    D = pairwise_distances(X, metric=metric)
    Ds = np.sort(D, axis=1)
    d1 = Ds[:, 0]
    d2 = Ds[:, 1] if N >= 3 else d1.copy()
    return d1, d2

def compute_radii(d1, d2, policy="second", scale=1.0, offset=0.0, rmin=None, rmax=None):
    r = scale * (0.5 * (d1 + d2) if policy == "mean2" else d2) + offset
    if rmin is not None: r = np.maximum(r, rmin)
    if rmax is not None: r = np.minimum(r, rmax)
    return r

def largest_non_outlier_label(labels):
    uniq, cnts = np.unique(labels, return_counts=True)
    pool = {lab: n for lab, n in zip(uniq.tolist(), cnts.tolist()) if lab >= 0}
    return max(pool, key=pool.get) if pool else None

def choose_cluster(centroids, labels, img_shape, mode="center"):
    uniq = [u for u in np.unique(labels) if u >= 0]
    if not uniq:
        return centroids  # nur Ausreißer -> alles nehmen
    if mode == "largest":
        lab = largest_non_outlier_label(labels)
        return centroids[labels == lab]
    # mode == "center"
    H, W = img_shape[:2]
    center = np.array([W / 2.0, H / 2.0])
    best_lab, best_d = None, np.inf
    for lab in uniq:
        pts = centroids[labels == lab]
        if len(pts) == 0: continue
        m = pts.mean(axis=0)
        d = np.linalg.norm(m - center)
        if d < best_d:
            best_d, best_lab = d, lab
    return centroids[labels == best_lab]

def overlay_bgra_on_bgr(bg_bgr: np.ndarray, fg_bgra: np.ndarray) -> np.ndarray:
    """Legt ein BGRA-Overlay (mit Alpha) über ein BGR-Bild. Skaliert das Overlay bei Bedarf."""
    Hb, Wb = bg_bgr.shape[:2]
    Hf, Wf = fg_bgra.shape[:2]
    if (Hb, Wb) != (Hf, Wf):
        fg_bgra = cv2.resize(fg_bgra, (Wb, Hb), interpolation=cv2.INTER_LINEAR)
        Hf, Wf = Hb, Wb
    alpha = fg_bgra[:, :, 3:4].astype(np.float32) / 255.0  # (H,W,1)
    fg_bgr = fg_bgra[:, :, :3].astype(np.float32)
    bg = bg_bgr.astype(np.float32)
    out = alpha * fg_bgr + (1.0 - alpha) * bg
    return out.astype(np.uint8)

# =========================
# Pipeline
# =========================

def main():
    try:
        log(f"PARAMS: DBSCAN_EPS={DBSCAN_EPS}, AREA_P_LOW={AREA_P_LOW}, CLUSTER_SELECT={CLUSTER_SELECT}")
        log("[1/9] Lade Bild (Mask2) …")
        img = cv2.imread(original_image_path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Bild nicht gefunden: {original_image_path}")

        log("[2/9] Vorverarbeitung (Graustufen, Blur, Adaptive Threshold) …")
        gray, thresh = preprocess(img)

        log("[3/9] Konturen, Zentren, Flächen & Circularity …")
        centroids_all, areas_all, circs_all, _ = contours_centroids_area_circularity(thresh)
        log(f"    Roh-Zentren: {len(centroids_all)}")
        if len(centroids_all) == 0:
            log("Keine Zentren gefunden. Abbruch.")
            return
        for p in [25, 50, 75, 90, 95, 99]:
            log(f"    Area p{p}: {np.percentile(areas_all, p):.1f} px²")

        log("[4/9] Größen-/Form-Filter anwenden …")
        centroids_f, mask_kept = apply_size_shape_filter(centroids_all, areas_all, circs_all)
        log(f"    Nach Filter: {len(centroids_f)}")
        if len(centroids_f) == 0:
            log("Nach Size/Shape-Filter keine Punkte übrig. Schwellen anpassen.")
            return

        log("[5/9] DBSCAN auf gefilterten Punkten …")
        db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES).fit(centroids_f.astype(np.float32))
        labels = db.labels_
        kept_pts = choose_cluster(centroids_f, labels, img.shape, mode=CLUSTER_SELECT)
        if len(kept_pts) == 0:
            log("Keine Punkte im gewählten Cluster. Parameter prüfen.")
            return
        log(f"    Gewählte Punkte für Kreise: {len(kept_pts)} ({CLUSTER_SELECT}-Selektion)")

        log("[6/9] Abstände (d1, d2) & Radien berechnen …")
        d1, d2 = two_nn_distances(kept_pts, metric=DIST_METRIC)
        radii = compute_radii(d1, d2, policy=RADIUS_POLICY,
                              scale=SCALE, offset=OFFSET,
                              rmin=MIN_RADIUS, rmax=MAX_RADIUS)

        log("[7/9] Bilder rendern & speichern (Overlay und Transparent-PNG) …")
        # Overlay (JPG) mit Umriss-Kreisen
        overlay = img.copy()
        for (cx, cy), r in zip(kept_pts, radii):
            r_int = int(round(r))
            if r_int > 0:
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), r_int, EDGE_COLOR_BGR, EDGE_THICKNESS)
        ok1 = safe_imwrite(out_jpg, overlay)

        # PNG (transparent) mit gefüllten Kreisen
        rgba = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        if DRAW_FILLED_ON_PNG:
            for (cx, cy), r in zip(kept_pts, radii):
                r_int = int(round(r))
                if r_int > 0:
                    cv2.circle(mask, (int(round(cx)), int(round(cy))), r_int, 255, -1)
            rgba[mask == 0] = (0, 0, 0, 0)
            for (cx, cy), r in zip(kept_pts, radii):
                r_int = int(round(r))
                if r_int > 0:
                    cv2.circle(rgba, (int(round(cx)), int(round(cy))), r_int, FILL_COLOR_BGRA, -1)
        ok2 = safe_imwrite(out_png, rgba)

        if not ok1 or not ok2:
            raise IOError(f"Speichern fehlgeschlagen: overlay_ok={ok1}, rgba_ok={ok2}")

        log("[8/9] Transparentes PNG über Original2 legen …")
        bg_path = resolve_image_path(background_image_hint)
        if bg_path is None:
            raise FileNotFoundError(f"Original2-Bild nicht gefunden (Präfix): {background_image_hint}")
        bg = cv2.imread(bg_path, cv2.IMREAD_COLOR)
        if bg is None:
            raise FileNotFoundError(f"Konnte Original2 nicht laden: {bg_path}")

        # PNG (RGBA) ggf. erneut laden (stellt sicher, dass wir die Alpha-Version nehmen)
        fg = cv2.imread(out_png, cv2.IMREAD_UNCHANGED)
        if fg is None or fg.shape[2] != 4:
            # falls nicht als BGRA geladen wurde, nehmen wir die in RAM erzeugte Variante
            fg = rgba

        composed = overlay_bgra_on_bgr(bg, fg)
        ok3 = safe_imwrite(out_all, composed)
        if not ok3:
            raise IOError("Speichern von 'alles_zusammen.png' fehlgeschlagen.")

        # Tabelle (nur fürs gewählte Set)
        log("[9/9] Tabelle ausgeben …")
        table = PrettyTable()
        table.field_names = ["Index", "X", "Y", "d1 (px)", "d2 (px)", "Radius (px)"]
        for i, ((cx, cy), a, b, r) in enumerate(zip(kept_pts, d1, d2, radii)):
            table.add_row([i, int(round(cx)), int(round(cy)), f"{a:.2f}", f"{b:.2f}", f"{r:.2f}"])
        print(table)

        log("✓ FERTIG")
        log(f"    Gespeichert:\n"
            f"    - Overlay (JPG): {out_jpg}\n"
            f"    - Transparent PNG: {out_png}\n"
            f"    - Alles zusammen (PNG): {out_all}")

        try:
            import winsound
            winsound.MessageBeep()
        except Exception:
            pass

    except Exception as e:
        log(f"FEHLER: {e}")

if __name__ == "__main__":
    main()
